# nano Beatrix — the twins trained together

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pair.ipynb)

The question: what happens when Castor and Pollux learn from each other while they train? Each pair trains both twins in one process on the very same rows, each with its own Adam and the same schedule, plus a cross term between their next-byte predictions. The term at a byte is the KL from one twin's distribution (held fixed) to the other's, weighted like that byte's own target.

| form | who learns from whom | at which bytes | why |
|---|---|---|---|
| 1 | Pollux from Castor | the asked values' bytes (the lookups' answers) | aimed at Pollux's weak point: at the 1,024-byte frame it finds the asked record in 31-41% of trials, Castor in 72-87% |
| 2 | Pollux from Castor | every byte | Castor's whole next-byte distribution, the text as well |
| 3 | each from the other | every byte | mutual learning in its published form |

The control is the v4 train itself (2026-10-13): the same seeds, the same rows and the same first-stage and first long-window block, each twin alone. Both twins of a pair start from the run's seed, so a pair with the cross term at zero trains each twin exactly as it trained alone; the trainer's CPU proof gives the weights byte for byte and every record field the same. In forms 1 and 2 Castor takes no cross term and trains as alone: its record against the train's is a repeat of the same run on this card (the card's repeat noise, measured for free).

Each pair runs the train's first stage (12,000 steps at 1,024 x 32, eight records a frame, the scrutiny rows, sliced frames) and then one long-window block (4,000 steps at 4,096 x 8, sixteen records a frame) continued from the paired crafts, the cross term on in both. Seed 1 is the Pollux that never formed its copy alone (two first-stage blocks); seed 2 formed. A pair costs about what its two twins cost alone: about 20 minutes on the RTX PRO 6000; a form takes about 40 minutes for both seeds; the three forms about 2 hours. Every finished pair ships both records and both weights to the training repo as it lands. C4 reads every pair against the train's solo blocks, and reads how alike the twins end inside (the KL each way, how often their next bytes agree, the debiased CKA of their states block by block) for the pairs and for the solo twins alike.

Put a Hugging Face token with write access to the training repo in this notebook's secrets as `HF_TOKEN` before C1; tokenless, everything runs and nothing ships. A `STOP` file in `/content/out` starts no new pair; rerunning C3 continues from the pairs on disk.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.24"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 24), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished pair ships here (both records, both weights); tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (every pass used these 160 MB)
ARM, PAIR = 'castor-both', 'pollux-both'               # the run's arm teaches in the one-way forms; the partner learns
SEEDS = [1, 2]                     # the train's seeds: seed 1's Pollux never formed alone (two first-stage blocks), seed 2's formed
FORMS = [                          # the cross term: xc 'one' = the partner learns from the arm, 'mutual' = each from the other
    dict(xc='one', xc_pos='value'),     # 1: Pollux from Castor at the asked values' bytes
    dict(xc='one', xc_pos='all'),       # 2: Pollux from Castor at every byte
    dict(xc='mutual', xc_pos='all'),    # 3: each from the other at every byte
]
XC_W = 1.0                         # a byte's cross term weighs like its own target (mutual learning's published weight)
XC_FROM = 0                        # on from the first step: the copy formed or failed between steps 1,800 and 3,000 in the train
BASE = dict(                       # the v4 train's rows (nano_v4_train.ipynb, FORM 'sliced')
    register='caption', frames='dyn', plant=0.75, ask='any', tags='mode', gauge_asks='nested,overlap', log_every=100,
    sized=1, sliced=32)
STAGE_A = dict(ctx=1024, mb=32, steps=12000, sizes='32,128,256', frame_records=8, nest=0.6, overlap=0.3, stage='A1')    # the train's stage A
STAGE_B = dict(ctx=4096, mb=8, steps=4000, sizes='32,128,256,512,1024', frame_records=16, nest=0.6, overlap=0.3, stage='B1')   # its stage B
print('decision block: pairs', ARM, '+', PAIR, '| seeds', SEEDS, '| forms', [f"{f['xc']}/{f['xc_pos']}" for f in FORMS], '| weight', XC_W,
      '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, every pair the notebook runs (its two records' names), the solo controls on the Hub, the push probe.
import json, os, time, torch
from huggingface_hub import HfApi
from geolip.nano_beatrix import bed_corpus, bed_runner
from geolip.nano_beatrix.names import run_tag, partner_args

if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)       # tokenless; a few minutes; the lines carry the time left
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

def tags(spec):
    # the two records a pair spec writes, per seed: (the arm's tag, the partner's tag), from the very arguments the runner passes
    out = {}
    for r in bed_runner.expand(spec):
        kv = dict(a.lstrip('-').split('=', 1) for a in r)
        out[int(kv['seed'])] = (run_tag(kv), run_tag(partner_args(kv)))
    return out

def specs(f):
    # one form's two grids: the first stage from birth, then the long-window block from each seed's paired crafts
    xc = dict(pair=PAIR, xc=f['xc'], xc_pos=f['xc_pos'], xc_w=XC_W, xc_from=XC_FROM)
    a = dict(BASE, **STAGE_A, **xc, arms=ARM, seeds=','.join(map(str, SEEDS)))
    ta = tags(a)
    b = [dict(BASE, **STAGE_B, **xc, arms=ARM, seeds=str(s), init=ta[s][0], pair_init=ta[s][1]) for s in SEEDS]
    return a, b

api = HfApi(token=False)
on_hub = set(api.list_repo_files(REPO))
plan = []
for i, f in enumerate(FORMS, 1):
    a, bs = specs(f)
    for s, (t, p) in tags(a).items():
        plan.append((i, 'A', s, t, p))
    for b in bs:
        for s, (t, p) in tags(b).items():
            plan.append((i, 'B', s, t, p))
for i, st, s, t, p in plan:
    done = f'bed/reports/bed_{t}.json' in on_hub
    print(f'form {i} stage {st} seed {s}: {"ON THE HUB " if done else ""}{t}  +  {p}')
    assert len(f'bed/reports/bed_{p}.json') < 200, 'a record name too long for the mind'

# the solo controls: the train's own blocks of the same seeds (the pair tags without their pair piece)
import re
solo = sorted({re.sub(r'_pair-[a-z]+-[a-z]+-[tsm][va](w[0-9.e-]+)?(f[0-9]+)?', '', n) for _, _, _, t, p in plan for n in (t, p)})
missing = [t for t in solo if f'bed/reports/bed_{t}.json' not in on_hub]
print(len(solo), 'solo controls from the train;', 'all on the Hub' if not missing else f'MISSING {missing}')

PUSH_OK = False
if HF_TOKEN:                                           # the ship-complete law: a write and a read-back BEFORE any training
    api_w = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_nano_pair.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'forms': FORMS, 'seeds': SEEDS, 'xc_w': XC_W, 'xc_from': XC_FROM, 'base': BASE}).encode()
    api_w.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano_pair: push probe')
    PUSH_OK = path in set(api_w.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')

In [ ]:
# C3 (T) — THE PAIRS: form by form, the first stage for both seeds, then each seed's long-window block from its paired crafts. One pair
# at a time on the card; both records and both weights ship in the background as each pair lands (a Hub fault retries and never stops
# the run). A STOP file in OUT starts no new pair. Rerun the cell to continue: the pairs on disk are read, not rerun (a new session
# fetches each long-window block's two starting crafts from the Hub). An interrupt loses only the running pair.
push = REPO if (HF_TOKEN and PUSH_OK) else ''
for i, f in enumerate(FORMS, 1):
    a, bs = specs(f)
    print(f'===== form {i}: xc {f["xc"]} at {f["xc_pos"]}: the first stage =====')
    res = bed_runner.run(a, out=OUT, push=push)
    assert res['failed'] is None and not res['stopped'], f'form {i} first stage: {res}'
    for b in bs:
        print(f'===== form {i}: the long-window block, seed {b["seeds"]} =====')
        res = bed_runner.run(b, out=OUT, push=push)
        assert res['failed'] is None and not res['stopped'], f'form {i} long-window block: {res}'
print('all pairs done')

In [ ]:
# C4 — THE READ: every pair against the train's solo blocks of the same seeds, and how alike the twins end inside (paired and solo
# alike: the solo twins' weights are fetched from the Hub and read on the same held-out rows). Run any time; it trains nothing.
from IPython.display import Markdown, display
from geolip.nano_beatrix import bed_pair
rows = [(f"{i}: {f['xc']} / {f['xc_pos']}", st, s, t, p) for i, f in enumerate(FORMS, 1) for st, spec in (('A', specs(f)[0]),) + tuple(('B', b) for b in specs(f)[1])
        for s, (t, p) in tags(spec).items()]
display(Markdown(bed_pair.report(rows, out=OUT, repo=REPO)))

### Reading the table

Each row is one twin after one stage: alone (the train's block) or paired (the form). The first stage reads the copy (the whole value pooled over every lookup group, and the first step the near records' bits went under the uniform rate: how early the copy formed), the nested and redefined-key questions (192 rows each) and the text. The long-window block reads the 1,024-byte frame twice: how often the first letter is right (finding the asked record) and how often the whole value is (finding and completing it), and the text. The last columns read how alike the two twins are inside on 32 held-out documents: the KL between their next-byte distributions each way, how often their next bytes agree, and the debiased CKA of their states at the middle and the last block (1 = the same up to a rotation and a scale).

| to | the edit in C1 |
|---|---|
| one form only | `FORMS = [dict(xc='one', xc_pos='value')]` |
| a lighter or heavier cross term | `XC_W = 0.5` or `XC_W = 2.0` (the names carry the weight) |
| the term from a later step | `XC_FROM = 2000` |
| more seeds | `SEEDS = [1, 2, 3, 4]` (seeds 3 and 4 have no solo control in the train; pass four's are plain-row crafts) |